In [1]:
import pennylane as qp
import pennylane.estimator as qre
import numpy as np
import scipy
import matplotlib.pyplot as plt
import logging
from collections import defaultdict
from pennylane.resource import SpectralNormError


from scipy.stats import rv_continuous
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)

from sympy import simplify, latex, symbols, init_printing, Matrix, eye, conjugate as conj, MatMul as mm, sqrt, Rational
from sympy.physics.quantum import TensorProduct as tp, Ket
init_printing()

np.random.seed(4)
qp.numpy.random.seed(4)

## QVST vs Trotter
Why QSVT with QROM vs Trooterization 

Hubbard is local and structured
QSVT/QROM is especially powerful when the Hamiltonian is given as a large irregular table: But the Hubbard model on a regular lattice is not an arbitrary table. It has simple repeated structure:

- nearest-neighbor hopping,
- osite interaction,
- mostly uniform coefficients,
- sparse local geometry.


# Goal

- Practical resource estimation for early-to-intermediate fault-tolerant quantum simulation of strongly correlated materials. Choose 3D Hubbard model using PennyLane's Labs estimator
- Give systematic, empirical comparison of worst-case vs. observable-specific Trotter error bounds
- Able to distinguish asymptotic theory from realistic hardware constraints;
- Capable of producing technically honest estimates that are useful for strategic decision-making inside an FTQC organization.

# Sub goal
- Aura farming with QROM?

# Create the Hamiltonian

In [2]:
t = 1.0
U = 4.0
n_cells = [2,2,1]
time = 5

In [3]:

H = qp.spin.fermi_hubbard(
    lattice="cubic",
    n_cells=n_cells,
    hopping=t,
    coulomb=U,
    boundary_condition=False,
    mapping="jordan_wigner"
)
n_qubits = H.num_wires

This link is interesting https://fermi-hubbard-commutators.readthxxedocs.io/en/latest/commutator_bounds.html

## Comparison
1st vs 2nd and 4th order Trotter-Suzuki

How badly do standard Trotter error bounds overestimate cost for specific observables in Hubbard, and what's the practical implication for resource estimates?"

Idea is, how to not actually have to run the simulation, but can estimate the error as close as possbile

1. Calculate the actual $\exp(-itH)$
2. Naive / Pessimistic estimator
3. How can we do better?
    - Method 1: BCH expansion
    - Method 2: Spectral Norm (Demo here https://pennylane.ai/qml/demos/tutorial_error_prop)
    - Method 3: Try to order the Hamiltonian such that consecutive terms commutator is as close to 0 as possible
        - How to measure "close to zero", matrix norm?
        - Complexity, there are $n!$ way of combination?
        - May need to leverage the properties of sparse and local Hamiltonian of Hubbard

## A view of error Worst-case vs. typical-case
The spectral norm $\lVert {U_{exact} - U_{trotter}}\rVert_2$ measures the maximum possible deviation over all possible input states in the entire Hilbert space.

However in common application, these relevant states (e.g, low-energy excited states with specific symmetries) is in a much smaller space. Therefore it is not the best interest to tune the Trotter using the Spectral Norm.

The next natural question is for the Hubbard model, the most relavant states are? In this example we use half-filled Neel state ($\ket{0101...01}$), which is standard for studying Mott insulators.

Haar-random states are commonly uses in quantum algorithm analysis because if its uniformly random over the entire Hilbert space. However have no structure relevant to the lattice, fermions, or interactions. Hubbard physics is about local correlations, hopping, double occupancy penalties, and half-filling.

# Classical error estimation method

They are about the operator itself in general Hilber space. Spectral norm is “pessimistic” because it answers the question

## Spectral norm error

$$ \bbox[yellow]
{
e^x=\lim_{n\to\infty} \left( 1+\frac{x}{n} \right)^n
\qquad (1)
}
$$

In [4]:


exact_op = qp.exp(H, 1j * time)
approx_op = qp.TrotterProduct(H, time, order=2)

error = SpectralNormError.get_error(exact_op, approx_op)
print(f"Spectral nor error algorithm: {error:.5f}")

Spectral nor error algorithm: 1.99526


Copy verbatim from Pennylane tutorial, 
#### todo change
> In general, exactly computing the spectral norm is computationally expensive for larger systems as it requires diagonalizing the operators. For this reason, we typically use upper bounds on the spectral norm error in the product formulas.

> We provide two common methods for bounding the error from literature [1]. They can be accessed by using op.error() and specifying the method keyword argument:

## Childs method

In [5]:
# op = qp.TrotterProduct(H, time, order=2)

# one_norm_error_bound = op.error(method="one-norm-bound")
# commutator_error_bound = op.error(method="commutator-bound")

# print("one-norm bound:   ", one_norm_error_bound)
# print("commutator bound: ", commutator_error_bound)

Most of the method below isn't too relevant for physics application. We care more about state fidelity

# Hubbard specialized method

## State initialization

Neel state ($\ket{0101...01}$) is the standard state for studying ferromagnetics problem

I set the state as the eigenstate from H. This way we can compare the accuracy of different methods.

We use the following config:

| Spin Orbital | Represents     |
|--------------|----------------|
| 0            | site 1, spin ↑ |
| 1            | site 1, spin ↓ |
| 2            | site 2, spin ↑ |
| 3            | site 2, spin ↓ |

In [6]:
def prepare_neel_state(wires):
    """Néel state: alternating up/down on bipartite lattice"""
    for site in range(wires // 2):
        if site % 2 == 0:
            qp.PauliX(wires=2 * site)
        else:
            qp.PauliX(wires=2 * site + 1)

# With circuit

In [7]:
dev = qp.device("default.qubit")


@qp.qnode(dev)
def exact_circ(H, t, wires):
    """
    Simluate exact evolution
    """
    initial_state = prepare_neel_state(wires)
    qp.exp(H, t)
    return qp.state()


@qp.qnode(dev)
def trotter_circ(H, t, wires):
    initial_state = prepare_neel_state(wires)
    qp.TrotterProduct(H, t)
    return qp.state()

In [8]:
exact_state = exact_circ(H, time, n_qubits)
trotter_state = trotter_circ(H, time, n_qubits)
print(qp.math.fidelity_statevector(exact_state, trotter_state))


errors_dict = qp.resource.algo_error(trotter_circ)(H, time, n_qubits)
print(errors_dict)

6.750704899154381e+39
{'SpectralNormError': SpectralNormError(3600.0)}


In [9]:
errors_dict

{'SpectralNormError': SpectralNormError(3600.0)}

## Business output

If we increase the order based on the Spectral norm estimation, then the number of gates vs accuracy looks like this

But if we increase the order based on the state fidelity, then the number of gates vs accuracy looks like this




In [10]:
# !pip install pennylane --target=/kaggle/working/